# Retention Modeling Dataset Validation

This notebook validates the employee-level dataset created for attrition prediction.

The modeling design uses:

- Snapshot date: June 30, 2025
- Prediction window: July 1, 2025 through June 30, 2026
- Target: `attrition_next_12m`

The dataset contains one row per employee who was employed on the snapshot date.

Only information available on or before the snapshot date is used to construct historical features.

Direct outcome columns such as employment status, termination date, and termination type are excluded to reduce target leakage.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "retention_modeling_dataset.csv"
)


retention = pd.read_csv(
    DATA_PATH,
    parse_dates=[
        "snapshot_date",
        "prediction_end_date",
    ],
)


print(
    "Dataset shape:",
    retention.shape,
)

Dataset shape: (7386, 39)


In [2]:
retention.head(10)

,employee_id,snapshot_date,prediction_end_date,approx_age,tenure_years,employment_type,education_level,application_source,years_experience_at_hire,hire_department_name,...,in_progress_training_programs,completed_training_hours,average_training_score,prior_promotion_events,prior_transfer_events,prior_manager_change_events,prior_leave_events,prior_change_events,days_since_last_change_event,attrition_next_12m
0,100001,2025-06-30,2026-06-30,40,2.70,Salaried,Bachelor's,Company Careers Page,12.0,Engineering,...,0,45.2,83.70,0,0,0,0,0,NaN,0
1,100002,2025-06-30,2026-06-30,54,3.24,Salaried,Master's,Employee Referral,12.0,Engineering,...,0,55.4,84.25,0,1,0,2,3,69.0,0
2,100003,2025-06-30,2026-06-30,58,4.01,Salaried,High School,Indeed,10.0,Engineering,...,0,51.0,87.33,0,0,0,0,0,NaN,0
3,100004,2025-06-30,2026-06-30,59,3.27,Salaried,Master's,LinkedIn,10.0,Engineering,...,1,35.8,89.05,0,0,0,0,0,NaN,0
4,100005,2025-06-30,2026-06-30,43,2.14,Salaried,Master's,Company Careers Page,8.0,Engineering,...,0,60.8,90.50,0,0,0,0,0,NaN,0
5,100006,2025-06-30,2026-06-30,50,4.46,Salaried,Master's,Indeed,21.0,Engineering,...,2,35.3,88.45,0,0,0,0,0,NaN,0
6,100007,2025-06-30,2026-06-30,47,2.59,Salaried,Doctorate,Company Careers Page,8.0,Engineering,...,1,24.4,91.70,0,0,0,0,0,NaN,0
7,100008,2025-06-30,2026-06-30,36,2.61,Salaried,Associate,LinkedIn,21.0,Engineering,...,0,61.4,89.15,0,0,0,0,0,NaN,0
8,100009,2025-06-30,2026-06-30,58,2.57,Salaried,Master's,LinkedIn,18.0,Engineering,...,0,74.1,86.85,0,1,0,0,1,310.0,0
9,100010,2025-06-30,2026-06-30,36,3.79,Salaried,Associate,Employee Referral,19.0,Engineering,...,0,57.6,84.40,0,0,0,0,0,NaN,0


In [3]:
retention.info()

<class 'pandas.DataFrame'>
RangeIndex: 7386 entries, 0 to 7385
Data columns (total 39 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   employee_id                     7386 non-null   int64         
 1   snapshot_date                   7386 non-null   datetime64[us]
 2   prediction_end_date             7386 non-null   datetime64[us]
 3   approx_age                      7386 non-null   int64         
 4   tenure_years                    7386 non-null   float64       
 5   employment_type                 7386 non-null   str           
 6   education_level                 7386 non-null   str           
 7   application_source              7386 non-null   str           
 8   years_experience_at_hire        7386 non-null   float64       
 9   hire_department_name            7386 non-null   str           
 10  hire_region                     7386 non-null   str           
 11  hire_job_family

## 1. Primary-key checks

In [4]:
employee_checks = pd.Series(
    {
        "employee IDs are complete": (
            retention[
                "employee_id"
            ].notna().all()
        ),

        "employee IDs are unique": (
            retention[
                "employee_id"
            ].is_unique
        ),

        "dataset is not empty": (
            len(retention)
            > 0
        ),
    },
    name="passed",
)

employee_checks

employee IDs are complete    True
employee IDs are unique      True
dataset is not empty         True
Name: passed, dtype: bool

## 2. Snapshot-date checks

In [5]:
date_checks = pd.Series(
    {
        "snapshot date is correct": (
            retention[
                "snapshot_date"
            ]
            .eq(
                pd.Timestamp(
                    "2025-06-30"
                )
            )
            .all()
        ),

        "prediction end date is correct": (
            retention[
                "prediction_end_date"
            ]
            .eq(
                pd.Timestamp(
                    "2026-06-30"
                )
            )
            .all()
        ),
    },
    name="passed",
)

date_checks

snapshot date is correct          True
prediction end date is correct    True
Name: passed, dtype: bool

## 3. Attrition-target checks

In [6]:
target_checks = pd.Series(
    {
        "target has no missing values": (
            retention[
                "attrition_next_12m"
            ].notna().all()
        ),

        "target contains only zero and one": (
            set(
                retention[
                    "attrition_next_12m"
                ]
            )
            .issubset(
                {
                    0,
                    1,
                }
            )
        ),

        "dataset contains retained employees": (
            (
                retention[
                    "attrition_next_12m"
                ]
                == 0
            ).any()
        ),

        "dataset contains attrition employees": (
            (
                retention[
                    "attrition_next_12m"
                ]
                == 1
            ).any()
        ),
    },
    name="passed",
)

target_checks

target has no missing values            True
target contains only zero and one       True
dataset contains retained employees     True
dataset contains attrition employees    True
Name: passed, dtype: bool

## 4. Attrition class balance

In [7]:
target_distribution = (
    retention[
        "attrition_next_12m"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "attrition_next_12m"
    )
    .reset_index(
        name="employee_count"
    )
)

target_distribution

,attrition_next_12m,employee_count
0,0,6759
1,1,627


In [8]:
attrition_rate = (
    retention[
        "attrition_next_12m"
    ].mean()
    * 100
)

print(
    f"Attrition rate: "
    f"{attrition_rate:.2f}%"
)

Attrition rate: 8.49%


## 5. Target-leakage checks

In [9]:
leakage_columns = [
    "employment_status",
    "termination_date",
    "termination_type",
]

present_leakage_columns = [
    column
    for column
    in leakage_columns
    if column
    in retention.columns
]

print(
    "Leakage columns found:",
    present_leakage_columns,
)

Leakage columns found: []


In [10]:
assert (
    len(
        present_leakage_columns
    )
    == 0
)

print(
    "No direct target-leakage "
    "columns detected."
)

No direct target-leakage columns detected.


## 6. Missing-value analysis

In [11]:
missing_summary = (
    retention
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .to_frame(
        "missing_count"
    )
)

missing_summary[
    "missing_percent"
] = (
    100
    * missing_summary[
        "missing_count"
    ]
    / len(retention)
)

missing_summary[
    missing_summary[
        "missing_count"
    ]
    > 0
]

,missing_count,missing_percent
days_since_last_change_event,5542,75.033848
average_performance_rating,1805,24.438126
promotion_recommended,1805,24.438126
performance_rating,1805,24.438126
days_since_review,1805,24.438126
goal_completion,1805,24.438126
average_training_score,124,1.678852


## 7. Numerical feature summary

In [12]:
numerical_columns = [
    "approx_age",
    "tenure_years",
    "years_experience_at_hire",
    "initial_base_salary",
    "base_salary",
    "salary_growth_percent",
    "performance_rating",
    "goal_completion",
    "completed_training_hours",
    "prior_promotion_events",
    "prior_transfer_events",
    "prior_leave_events",
]

retention[
    numerical_columns
].describe().T

,count,mean,std,min,25%,50%,75%,max
approx_age,7386.0,42.368400,11.872367,22.0,32.00,42.00,52.00,63.00
tenure_years,7386.0,2.233554,1.293443,0.0,1.11,2.20,3.36,4.49
years_experience_at_hire,7386.0,7.962632,4.817730,0.0,4.00,8.00,12.00,30.00
initial_base_salary,7386.0,85290.712158,26174.558194,47500.0,69800.00,77900.00,104100.00,187000.00
base_salary,7386.0,93280.016247,29368.487274,47700.0,74100.00,87800.00,112700.00,205200.00
salary_growth_percent,7386.0,9.383779,7.812491,0.0,3.03,8.56,14.78,38.46
performance_rating,5581.0,3.497223,0.515016,1.6,3.10,3.50,3.90,5.00
goal_completion,5581.0,92.020480,9.996574,47.0,85.20,91.80,98.90,120.00
completed_training_hours,7386.0,26.993840,17.772801,0.0,9.10,26.20,36.60,91.40
prior_promotion_events,7386.0,0.175332,0.418260,0.0,0.00,0.00,0.00,3.00


## 8. Categorical feature summary

In [13]:
categorical_columns = [
    "employment_type",
    "education_level",
    "application_source",
    "hire_department_name",
    "hire_region",
    "hire_job_family",
    "hire_job_level",
]

for column in categorical_columns:

    print(
        f"\n{column}"
    )

    print(
        retention[
            column
        ].value_counts(
            dropna=False
        )
    )


employment_type
employment_type
Salaried    6066
Hourly      1320
Name: count, dtype: int64

education_level
education_level
Bachelor's     3514
Master's       1609
Associate      1076
High School    1002
Doctorate       185
Name: count, dtype: int64

application_source
application_source
LinkedIn                    1825
Employee Referral           1629
Company Careers Page        1460
Indeed                       916
University Recruiting        609
Professional Association     377
Staffing Agency              373
Job Fair                     197
Name: count, dtype: int64

hire_department_name
hire_department_name
Manufacturing             1797
Engineering               1508
Supply Chain               900
Information Technology     756
Sales                      730
Finance                    585
Customer Support           578
Human Resources            532
Name: count, dtype: int64

hire_region
hire_region
West         3086
South        2073
Southwest    1323
Northeast     904
Name:

## 9. Initial attrition comparisons

In [14]:
attrition_by_employment_type = (
    retention
    .groupby(
        "employment_type"
    )
    .agg(
        employee_count=(
            "employee_id",
            "count",
        ),
        attrition_rate=(
            "attrition_next_12m",
            "mean",
        ),
    )
)

attrition_by_employment_type[
    "attrition_rate_percent"
] = (
    100
    * attrition_by_employment_type[
        "attrition_rate"
    ]
)

attrition_by_employment_type

,employee_count,attrition_rate,attrition_rate_percent
employment_type,,,
Hourly,1320,0.122727,12.272727
Salaried,6066,0.076657,7.665678


In [15]:
attrition_by_hire_department = (
    retention
    .groupby(
        "hire_department_name"
    )
    .agg(
        employee_count=(
            "employee_id",
            "count",
        ),
        attrition_rate=(
            "attrition_next_12m",
            "mean",
        ),
    )
    .sort_values(
        "attrition_rate",
        ascending=False,
    )
)

attrition_by_hire_department[
    "attrition_rate_percent"
] = (
    100
    * attrition_by_hire_department[
        "attrition_rate"
    ]
)

attrition_by_hire_department

,employee_count,attrition_rate,attrition_rate_percent
hire_department_name,,,
Manufacturing,1797,0.104619,10.461881
Customer Support,578,0.100346,10.034602
Information Technology,756,0.089947,8.994709
Finance,585,0.088889,8.888889
Sales,730,0.072603,7.260274
Human Resources,532,0.071429,7.142857
Supply Chain,900,0.071111,7.111111
Engineering,1508,0.070292,7.029178


## 10. Complete validation summary

In [16]:
all_checks = pd.concat(
    [
        employee_checks,
        date_checks,
        target_checks,
    ]
)

validation_results = pd.DataFrame(
    {
        "check": all_checks.index,
        "passed": all_checks.values,
    }
)

validation_results

,check,passed
0,employee IDs are complete,True
1,employee IDs are unique,True
2,dataset is not empty,True
3,snapshot date is correct,True
4,prediction end date is correct,True
5,target has no missing values,True
6,target contains only zero and one,True
7,dataset contains retained employees,True
8,dataset contains attrition employees,True


In [17]:
if validation_results[
    "passed"
].all():

    print(
        "All retention dataset "
        "validation checks passed."
    )

else:

    print(
        "One or more retention dataset "
        "validation checks failed."
    )

All retention dataset validation checks passed.


## 11. Conclusions

The retention modeling dataset successfully creates one employee-level record for each employee who was employed on June 30, 2025.

### Prediction design

- Snapshot date: June 30, 2025
- Prediction window: July 1, 2025 through June 30, 2026
- Target: `attrition_next_12m`

### Feature sources

The dataset combines historical information from:

- Employee records
- Recruiting applications
- Job requisitions
- Compensation history
- Performance reviews
- Training records
- Employee events

### Leakage controls

Direct termination information is not included as a feature.

Historical compensation, performance, training, and event features are restricted to information available on or before the snapshot date.

Original hiring department, region, job family, and job level are used instead of later organizational assignments.

### Missing data

Some employees may not have:

- A performance review before the snapshot
- A training assessment score
- A prior organizational change event

These missing values are expected and will be handled during machine-learning preprocessing.

### Next step

The next stage will prepare the feature matrix, encode categorical variables, split the data into training and testing sets, and build a baseline attrition-classification model.